# Agentic Teaching Copilot

In Week 1, this project created a basic Retrieval-Augmented Generation (RAG) pipeline for Applied Machine Learning course materials. In this notebook, we turn that pipeline into a small agent.

The agent can call tools over the course-material knowledge base. Instead of always retrieving context in a fixed chain, the model can decide when to search for relevant records and when to fetch the full content of a specific record.

This is an early prototype for Applied ML Teaching Copilot: an assistant for instructors and students that answers questions from course materials, explains concepts pedagogically, and avoids inventing unsupported content.

## 1. Setup

Load environment variables from `.env`, verify that `OPENAI_API_KEY` is available, and initialize the OpenAI client.

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not available. Add it to your .env file before running this notebook.")

openai_client = OpenAI()
print("OPENAI_API_KEY is available.")

OPENAI_API_KEY is available.


## 2. Load Course Materials

Each record should include `id`, `module`, `lesson`, `topic`, `content`, and `source_type`.

In [2]:
data_path = Path("data/course_materials.json")
if not data_path.exists():
    data_path = Path("../data/course_materials.json")

with data_path.open("r", encoding="utf-8") as f:
    documents = json.load(f)

required_fields = {"id", "module", "lesson", "topic", "content", "source_type"}
missing_fields = [
    (doc.get("id", "<missing id>"), sorted(required_fields - set(doc)))
    for doc in documents
    if required_fields - set(doc)
]

if missing_fields:
    raise ValueError(f"Some course-material records are missing required fields: {missing_fields}")

materials_by_id = {doc["id"]: doc for doc in documents}

print(f"Loaded {len(documents)} course-material records.")
documents[:2]

Loaded 10 course-material records.


[{'id': 'aml-001',
  'module': 'Regression',
  'lesson': 'Model Evaluation',
  'topic': 'MAE vs MSE',
  'content': 'Mean Absolute Error (MAE) averages the absolute size of prediction errors, so each error contributes linearly. Mean Squared Error (MSE) averages squared errors, so large mistakes receive much more weight. Use MAE when you want an evaluation metric that is easy to interpret in the target units and less sensitive to outliers. Use MSE when large errors are especially costly or when you want the training and evaluation process to strongly penalize big misses.',
  'source_type': 'lecture_note'},
 {'id': 'aml-002',
  'module': 'Supervised Learning',
  'lesson': 'Tree-Based Models',
  'topic': 'Decision trees',
  'content': 'A decision tree makes predictions by repeatedly splitting the data into smaller groups using feature-based rules. At each internal node, the tree chooses a split that makes the resulting groups more pure with respect to the target. Decision trees are useful 

## 3. Build a Search Index

The agent will use `minsearch` to search over the course-material topics and content, while keeping metadata fields available for filtering and display.

In [3]:
text_fields = ["topic", "content"]
keyword_fields = ["module", "lesson", "source_type", "id"]

index = Index(text_fields=text_fields, keyword_fields=keyword_fields)
index.fit(documents)

print("Built minsearch index.")

Built minsearch index.


## 4. Course-Material Tools

The first tool returns compact search results. The second tool retrieves a full record by `id`. Both tools write to `tool_call_history` so we can inspect what the agent did.

In [4]:
tool_call_history = []


def reset_tool_call_history():
    """Clear the notebook-level tool call history."""
    tool_call_history.clear()


def make_snippet(text: str, max_chars: int = 260) -> str:
    clean_text = " ".join(text.split())
    if len(clean_text) <= max_chars:
        return clean_text
    return clean_text[:max_chars].rsplit(" ", 1)[0] + "..."


def search_course_materials(query: str, num_results: int = 5) -> list[dict]:
    """
    Search the Applied Machine Learning course materials for records relevant to a user query.

    Args:
        query: The search query written by the user.
        num_results: Maximum number of search results to return.

    Returns:
        A list of matching course-material records with id, module, lesson, topic,
        source_type, and a short content snippet.
    """
    safe_num_results = max(1, min(int(num_results), 10))
    results = index.search(query, num_results=safe_num_results)

    compact_results = []
    for doc in results:
        compact_results.append({
            "id": doc["id"],
            "module": doc["module"],
            "lesson": doc["lesson"],
            "topic": doc["topic"],
            "source_type": doc["source_type"],
            "content_snippet": make_snippet(doc["content"]),
        })

    tool_call_history.append({
        "tool": "search_course_materials",
        "arguments": {"query": query, "num_results": safe_num_results},
        "result_count": len(compact_results),
    })

    return compact_results


def get_course_material(material_id: str) -> dict:
    """
    Retrieve the full content of a course-material record by id.

    Args:
        material_id: The id of the course-material record.

    Returns:
        The full course-material record if found, or an error dictionary if not found.
    """
    record = materials_by_id.get(material_id)

    tool_call_history.append({
        "tool": "get_course_material",
        "arguments": {"material_id": material_id},
        "found": record is not None,
    })

    if record is None:
        return {"error": f"No course material found with id '{material_id}'."}

    return record

## 5. Tool Schemas

These schemas tell the Responses API which Python tools are available and what arguments each tool expects.

In [5]:
tools = [
    {
        "type": "function",
        "name": "search_course_materials",
        "description": "Search the Applied Machine Learning course materials for records relevant to a user query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query written by the user.",
                },
                "num_results": {
                    "type": "integer",
                    "description": "Maximum number of search results to return.",
                    "default": 5,
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_course_material",
        "description": "Retrieve the full content of a course-material record by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "material_id": {
                    "type": "string",
                    "description": "The id of the course-material record to retrieve, such as aml-001.",
                },
            },
            "required": ["material_id"],
            "additionalProperties": False,
        },
    },
]

available_tools = {
    "search_course_materials": search_course_materials,
    "get_course_material": get_course_material,
}

tools

[{'type': 'function',
  'name': 'search_course_materials',
  'description': 'Search the Applied Machine Learning course materials for records relevant to a user query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query written by the user.'},
    'num_results': {'type': 'integer',
     'description': 'Maximum number of search results to return.',
     'default': 5}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'get_course_material',
  'description': 'Retrieve the full content of a course-material record by id.',
  'parameters': {'type': 'object',
   'properties': {'material_id': {'type': 'string',
     'description': 'The id of the course-material record to retrieve, such as aml-001.'}},
   'required': ['material_id'],
   'additionalProperties': False}}]

## 6. Agent Instructions

The agent should be helpful, grounded, and explicit when the course materials do not support an answer.

In [6]:
agent_instructions = """
You are Applied ML Teaching Copilot, an assistant for instructors and students working with Applied Machine Learning course materials.

Your job:
- Help instructors and students understand Applied Machine Learning concepts.
- Use search_course_materials when the user asks about a topic.
- Use get_course_material when you need full detail from a specific search result.
- Use only facts from the course materials when answering.
- Use only information returned by the tools.
- Never answer from general model knowledge.
- If search_course_materials returns zero results, say that the current course-material knowledge base does not contain enough information to answer.
- If no full course material is fetched with get_course_material, do not provide detailed explanations.
- For out-of-scope topics, suggest adding relevant material to the course knowledge base instead of explaining the topic from memory.
- Say clearly when the course materials do not contain enough information.
- Explain concepts clearly and pedagogically.
- Prefer concise, structured answers with examples when possible.
- Never pretend that a topic is covered if it is not in the retrieved material.

When answering, cite the relevant material ids in plain text, such as "Source: aml-001".
""".strip()

print(agent_instructions)

You are Applied ML Teaching Copilot, an assistant for instructors and students working with Applied Machine Learning course materials.

Your job:
- Help instructors and students understand Applied Machine Learning concepts.
- Use search_course_materials when the user asks about a topic.
- Use get_course_material when you need full detail from a specific search result.
- Use only facts from the course materials when answering.
- Use only information returned by the tools.
- Never answer from general model knowledge.
- If search_course_materials returns zero results, say that the current course-material knowledge base does not contain enough information to answer.
- If no full course material is fetched with get_course_material, do not provide detailed explanations.
- For out-of-scope topics, suggest adding relevant material to the course knowledge base instead of explaining the topic from memory.
- Say clearly when the course materials do not contain enough information.
- Explain concep

## 7. Custom Responses API Agent Loop

This loop sends a user question to the model, handles any function calls, executes the matching Python tools, appends `function_call_output` items, and continues until the model returns a final answer.

In [7]:
def response_item_to_dict(item):
    if hasattr(item, "model_dump"):
        return item.model_dump()
    return item


def should_force_insufficient_materials_answer() -> bool:
    return (
        len(tool_call_history) == 1
        and tool_call_history[0]["tool"] == "search_course_materials"
        and tool_call_history[0].get("result_count") == 0
    )


def insufficient_materials_answer(question: str) -> str:
    return (
        f"The current course materials do not contain enough information to answer this question using the course materials: {question} "
        "To support this query, we should add relevant course notes, slides, or readings to the knowledge base."
    )


def run_teaching_copilot(question: str, model: str = "gpt-4o-mini", max_turns: int = 8) -> dict:
    reset_tool_call_history()

    message_history = [
        {
            "role": "user",
            "content": question,
        }
    ]

    for _ in range(max_turns):
        response = openai_client.responses.create(
            model=model,
            instructions=agent_instructions,
            input=message_history,
            tools=tools,
            tool_choice="auto",
        )

        response_items = [response_item_to_dict(item) for item in response.output]
        message_history.extend(response_items)

        function_calls = [item for item in response.output if getattr(item, "type", None) == "function_call"]

        if not function_calls:
            final_answer = response.output_text
            if should_force_insufficient_materials_answer():
                final_answer = insufficient_materials_answer(question)

            return {
                "final_answer": final_answer,
                "tool_call_history": list(tool_call_history),
                "message_history": message_history,
            }

        for function_call in function_calls:
            tool_name = function_call.name
            arguments = json.loads(function_call.arguments or "{}")

            print(f"Tool call: {tool_name}({arguments})")

            tool_fn = available_tools.get(tool_name)
            if tool_fn is None:
                tool_result = {"error": f"Unknown tool: {tool_name}"}
            else:
                try:
                    tool_result = tool_fn(**arguments)
                except Exception as exc:
                    tool_result = {"error": f"Tool execution failed: {exc}"}

            message_history.append({
                "type": "function_call_output",
                "call_id": function_call.call_id,
                "output": json.dumps(tool_result),
            })

    raise RuntimeError("Agent did not finish within max_turns.")

## 8. Test the Agent

Run three queries. The first two should be answerable from the current course materials. The third asks about convolutional neural networks, which are likely outside the small prototype dataset; the agent should say the available materials are insufficient if it cannot find support.

In [8]:
query_a = "When should I use MAE instead of MSE in a regression problem?"
result_a = run_teaching_copilot(query_a)

print("\nFinal answer:\n")
print(result_a["final_answer"])

print("\nTool call history:")
result_a["tool_call_history"]

Tool call: search_course_materials({'query': 'MAE vs MSE in regression', 'num_results': 5})
Tool call: get_course_material({'material_id': 'aml-001'})

Final answer:

In regression problems, the choice between Mean Absolute Error (MAE) and Mean Squared Error (MSE) depends on your specific goals and the nature of your data:

### When to Use MAE:
- **Interpretability**: MAE is expressed in the same units as the target variable, making it easier to understand.
- **Sensitivity to Outliers**: MAE is less sensitive to outliers, as it treats all errors linearly. This means that a few large errors won't disproportionately affect the overall error metric.

### When to Use MSE:
- **Cost of Large Errors**: MSE penalizes larger errors more significantly due to the squaring of errors. This is suitable when larger errors are particularly undesirable or costly.
- **Emphasis on Performance**: If you want to enforce a stronger penalty during the training and evaluation process for larger mistakes, MSE 

[{'tool': 'search_course_materials',
  'arguments': {'query': 'MAE vs MSE in regression', 'num_results': 5},
  'result_count': 5},
 {'tool': 'get_course_material',
  'arguments': {'material_id': 'aml-001'},
  'found': True}]

In [9]:
query_b = "Give me a short study guide about decision trees."
result_b = run_teaching_copilot(query_b)

print("\nFinal answer:\n")
print(result_b["final_answer"])

print("\nTool call history:")
result_b["tool_call_history"]

Tool call: search_course_materials({'query': 'decision trees', 'num_results': 5})
Tool call: get_course_material({'material_id': 'aml-002'})
Tool call: get_course_material({'material_id': 'aml-004'})
Tool call: get_course_material({'material_id': 'aml-003'})

Final answer:

### Study Guide: Decision Trees

#### Overview
- **Definition**: A decision tree is a predictive model that categorizes data by splitting it into smaller groups based on feature-based rules. These splits aim to produce groups that are more homogeneous concerning the target variable.

- **Characteristics**:
  - **Interpretable**: Easy to understand and visualize.
  - **Non-linear Relationships**: Can capture complex interactions between features.
  - **Risk of Overfitting**: Very deep trees may fit the training data too closely and fail to generalize well to new data.

#### Key Concepts

1. **Splitting**:
   - At each internal node of the tree, a feature and a threshold are selected to split the data into two child n

[{'tool': 'search_course_materials',
  'arguments': {'query': 'decision trees', 'num_results': 5},
  'result_count': 3},
 {'tool': 'get_course_material',
  'arguments': {'material_id': 'aml-002'},
  'found': True},
 {'tool': 'get_course_material',
  'arguments': {'material_id': 'aml-004'},
  'found': True},
 {'tool': 'get_course_material',
  'arguments': {'material_id': 'aml-003'},
  'found': True}]

In [10]:
query_c = "Can you explain convolutional neural networks using the course materials?"
result_c = run_teaching_copilot(query_c)

print("\nFinal answer:\n")
print(result_c["final_answer"])

print("\nTool call history:")
result_c["tool_call_history"]

Tool call: search_course_materials({'query': 'convolutional neural networks', 'num_results': 5})

Final answer:

The current course materials do not contain enough information to answer this question using the course materials: Can you explain convolutional neural networks using the course materials? To support this query, we should add relevant course notes, slides, or readings to the knowledge base.

Tool call history:


[{'tool': 'search_course_materials',
  'arguments': {'query': 'convolutional neural networks', 'num_results': 5},
  'result_count': 0}]

## 9. Final Summary Table

This table summarizes the test queries, tools called, and a short version of each final answer.

In [11]:
def short_answer(text: str, max_chars: int = 160) -> str:
    clean_text = " ".join(text.split())
    if len(clean_text) <= max_chars:
        return clean_text
    return clean_text[:max_chars].rsplit(" ", 1)[0] + "..."


def markdown_escape(text: str) -> str:
    return text.replace("|", "\\|").replace("\n", " ")


summary_rows = []
for query, result in [
    (query_a, result_a),
    (query_b, result_b),
    (query_c, result_c),
]:
    tools_called = ", ".join(call["tool"] for call in result["tool_call_history"])
    summary_rows.append({
        "query": query,
        "tools_called": tools_called,
        "final_answer_short": short_answer(result["final_answer"]),
    })

print("| query | tools_called | final_answer_short |")
print("| --- | --- | --- |")
for row in summary_rows:
    print(
        "| "
        + markdown_escape(row["query"])
        + " | "
        + markdown_escape(row["tools_called"])
        + " | "
        + markdown_escape(row["final_answer_short"])
        + " |"
    )

| query | tools_called | final_answer_short |
| --- | --- | --- |
| When should I use MAE instead of MSE in a regression problem? | search_course_materials, get_course_material | In regression problems, the choice between Mean Absolute Error (MAE) and Mean Squared Error (MSE) depends on your specific goals and the nature of your data:... |
| Give me a short study guide about decision trees. | search_course_materials, get_course_material, get_course_material, get_course_material | ### Study Guide: Decision Trees #### Overview - **Definition**: A decision tree is a predictive model that categorizes data by splitting it into smaller groups... |
| Can you explain convolutional neural networks using the course materials? | search_course_materials | The current course materials do not contain enough information to answer this question using the course materials: Can you explain convolutional neural... |


## Homework form answers

### Q1: Planned tools

Planned tools for the Applied ML Teaching Copilot agent:

a) `search_course_materials` — essential
b) `get_course_material` — essential
c) `add_course_note` — future/nice to have
d) `generate_study_guide` — future/nice to have

### Q2: Implemented tool description

`search_course_materials` is the main implemented tool. It searches the Applied Machine Learning course-material knowledge base using a `minsearch` index over `topic` and `content`, with `module`, `lesson`, `source_type`, and `id` available as keyword fields. It returns compact records with source metadata and short snippets so the agent can decide whether the retrieved material is relevant. `get_course_material` was also implemented as a companion read/fetch tool for retrieving the full content of a specific course-material record by id.

### Q3: Agent instructions and one example interaction

The agent is instructed to act as an Applied Machine Learning Teaching Copilot for instructors and students. It should search when asked about a topic, fetch full material when needed, answer only from retrieved course materials, never answer from general model knowledge, cite material ids, and say when the current knowledge base is insufficient.

Real example interaction for Query A:

- User: "When should I use MAE instead of MSE in a regression problem?"
- Tool call: `search_course_materials({'query': 'MAE vs MSE regression', 'num_results': 5})`
- Tool call: `get_course_material({'material_id': 'aml-001'})`
- Final answer summary: use MAE when you want errors that are easy to interpret in target units and less sensitive to outliers; use MSE when large mistakes are especially costly or should be penalized more strongly. Source: `aml-001`.